# 35 (PW) — Federate & Govern

**Production workflow, step 4.** Cross-source federation and governance: register a foreign table, read it alongside IRIS data, and apply credential/session hygiene. Grounded in `docs/data_access.md` and `docs/security_guide.md`.

In [ ]:
import os
from dotenv import load_dotenv
from irispark import IrisParkSession

load_dotenv()

# Connection via environment variables (matches examples/basic_usage.py).
# Set IRIS_HOST / IRIS_PORT / IRIS_NAMESPACE / IRIS_USERNAME / IRIS_PASSWORD.
try:
    session = IrisParkSession.builder() \
        .host(os.environ.get("IRIS_HOST", "localhost")) \
        .port(int(os.environ.get("IRIS_PORT", 1972))) \
        .namespace(os.environ.get("IRIS_NAMESPACE", "USER")) \
        .username(os.environ.get("IRIS_USERNAME", "_SYSTEM")) \
        .password(os.environ.get("IRIS_PASSWORD", "SYS")) \
        .getOrCreate()
    print("Connected to IRIS:", session)
except Exception as e:
    print("SKIP: IRIS not reachable -", e)
    session = None

In [ ]:
if session is None:
    raise SystemExit("IRIS not reachable; skipping this notebook.")

## 1. Register a JDBC foreign table

`read.jdbc()` builds a foreign server + table. Password-in-DDL triggers a `UserWarning` — prefer a named `CONNECTION` in production.

In [ ]:
URL = "jdbc:IRIS://localhost:1972/DATASPARK"
try:
    ft = session.iris.register_jdbc_foreign_table(
        url=URL,
        dbtable="vendas",
        user="suser",
        password="pass123",
        driver="com.intersystems.jdbc.IRISDriver",
        name="pw_ft_vendas",
    )
    ft.show()
    print("registered:", type(ft).__name__)
except Exception as e:
    print("federation not available:", str(e)[:160])

## 2. Cross-source join

Join a foreign table with a local IRIS DataFrame — IRIS executes the join, data stays in IRIS.

In [ ]:
local = session.createDataFrame(
    [(1, "Sao Paulo"), (2, "Campinas")], ["id", "cidade_nome"]
)
try:
    joined = ft.join(local, "id").select("id", "cidade", "cidade_nome", "valor")
    joined.show()
    print("cross-source join ok")
except Exception as e:
    print("cross-source join not available:", str(e)[:120])

## 3. Write back through a foreign table

Publish a DataFrame to a remote table via the foreign server.

In [ ]:
try:
    session.createDataFrame([(99, "WriteBack", "SP", 1.0, "2025-01-01")],
                            ["id", "cidade", "estado", "valor", "data"]).write.jdbc(
        url=URL, dbtable="pw_ft_write", user="suser", password="pass123",
        driver="com.intersystems.jdbc.IRISDriver", mode="overwrite",
    )
    print("write-back ok")
except Exception as e:
    print("write-back not available:", str(e)[:120])

## 4. Governance & credential hygiene

Security best practice (`docs/security_guide.md`): prefer named `CONNECTION`s over inline passwords, enable TLS, and let foreign objects be session-scoped (dropped on `close()`).

In [ ]:
print("foreign objects are session-scoped and dropped on session.close()")

## 5. Session lifecycle

`close()` releases the connection and cleans up session-scoped foreign tables/servers.

In [ ]:
if session is not None:
    session.close()
    print("Session closed.")